# Intro to Context Caching with the Gemini API

## Overview

The Gemini API provides the context caching feature to store frequently used input tokens in a dedicated cache and use them for subsequent requests, eliminating the need to repeatedly pass the same set of tokens to a model. This feature can help reduce the number of tokens sent to the model, thereby lowering the cost of requests.

The Gemini API offers two different caching mechanisms:

- Implicit caching (automatic, no cost saving guarantee)
- Explicit caching (manual, cost saving guarantee)

For more information, refer to the [context caching overview](https://cloud.google.com/vertex-ai/generative-ai/docs/context-cache/context-cache-overview) page.

### Objectives

In this tutorial, you learn how to use implicit caching and explicit caching with the Google Gen AI SDK in Vertex AI.

You will complete the following tasks:

- Use implicit caching
- Use explicit caching
  - Create a context cache
  - Retrieve and use a context cache
  - Use context caching in Chat
  - Update the expire time of a context cache
  - Delete a context cache

## Get started

### Install Google Gen AI SDK for Python

In [1]:
#%pip install --upgrade --quiet google-genai

### Set Google Cloud project information and create client

To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [1]:
import os

# fmt: off
PROJECT_ID = "bdc-trainings"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
# fmt: on
LOCATION = "us-central1"  # @param {type:"string"}

if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "us-central1")

## Code Examples

### Import libraries

In [2]:
from IPython.display import Markdown, display
from google import genai
from google.genai.types import (
    Content,
    CreateCachedContentConfig,
    GenerateContentConfig,
    Part,
)

### Create a client

In [3]:
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

### Use a supported model

See context caching [supported models](https://cloud.google.com/vertex-ai/generative-ai/docs/context-cache/context-cache-overview#supported_models).

In [4]:
# fmt: off
MODEL_ID = "gemini-2.5-flash"  # @param ["gemini-3-flash-preview", "gemini-2.5-flash", "gemini-2.5-pro"] {"allow-input":true, isTemplate: true}
# fmt: on

## Implicit caching

Implicit caching directly passes cache cost savings to developers without the need to create an explicit cache. Now, when you send a request to one of the Gemini 2.5 models, if the request shares a common prefix as one of previous requests, then it's eligible for a cache hit.

**Note** that implicit caching is enabled by default for all Gemini 3 and 2.5 models but cost savings only apply to Gemini 2.5 models. The minimum input token count for implicit caching is 2,048 for 2.5 Flash and 2.5 Pro.

### Re-enable caching

By default, Google foundation models cache inputs for Gemini models. If you disabled caching and want to re-enable it, run the following curl command. To run this command, a user must be granted the Vertex AI administrator role `roles/aiplatform.admin`.

For more information about enabling and disabling data caching, see [documentation](https://cloud.google.com/vertex-ai/generative-ai/docs/data-governance#enabling-disabling-caching).

In [5]:
os.environ["API_ENDPOINT"] = (
    f"{LOCATION}-aiplatform.googleapis.com/v1/projects/{PROJECT_ID}"
)
os.environ["PROJECT_ID"] = PROJECT_ID

In [6]:
%%bash

# Enable caching
curl -X PATCH \
  -H "Authorization: Bearer $(gcloud auth print-access-token)" \
  -H "Content-Type: application/json" \
  https://${API_ENDPOINT}/cacheConfig \
  -d '{
    "name": "projects/${PROJECT_ID}/cacheConfig",
    "disableCache": false
  }'

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   366    0   285  100    81    152     43  0:00:01  0:00:01 --:--:--   195   670:00:01 --:--:--   195


{
  "name": "projects/456822750436/locations/us-central1/projects/456822750436/cacheConfig/operations/2413259153444175872",
  "done": true,
  "response": {
    "@type": "type.googleapis.com/google.cloud.aiplatform.v1.CacheConfig",
    "name": "projects/456822750436/cacheConfig"
  }
}


### Send a request with a large and common content

To increase the chance of an implicit cache hit:

- Put large and common contents at the beginning of your prompt
- Send requests with similar prefix in a short amount of time

In this example, you send a request with an image at the beginning of your prompt and add a user's request/question at the end of the prompt.

In [7]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        Part.from_uri(
            file_uri="https://storage.googleapis.com/cloud-samples-data/generative-ai/image/a-man-and-a-dog.png",
            mime_type="image/png",
        ),
        "Describe this image.",
    ],
)

display(Markdown(response.text))

A warm, close-up, selfie-style shot features a smiling man and an attentive dog posing together in what appears to be a brightly lit living room.

On the right side of the frame, the man looks directly at the camera, his arm partially visible on the far right, suggesting he is holding the camera. He has dark, curly hair, a full beard, and a mustache, and is wearing a blue, textured knit sweater. His expression is one of genuine happiness, with a wide smile and crinkled eyes.

To the man's left and slightly in front of him, a medium-sized dog with alert, perked-up ears also looks towards the camera. The dog has a mix of brown, black, and white fur, particularly around its muzzle and chest. Its expression is calm and engaged, with bright, intelligent eyes.

The background, though softly blurred, reveals details of a comfortable home environment. To the far left, a dark grey armchair with a striped cushion is visible, next to a tall wooden speaker. In the mid-ground, behind the dog, there's a table lamp with a light-colored shade. Behind the man, a soft grey sofa is adorned with a bright yellow throw pillow. Further back, natural light streams in from large windows, with one showing green foliage outside and another having vertical blinds. A wooden shelving unit and a dining table can also be seen in the distant background.

The overall impression is one of companionship and joy, capturing a moment of affection between the man and his dog.

### Send requests with similar prefixes

To demonstrate the implicit cache hit, repeat the same request multiple times, and print out the `cached_content_token_count` in the usage metadata which indicates how many tokens in the request were cached.

In [8]:
NUM_ATTEMPTS = 5  # @param {type: "integer"}

for i in range(NUM_ATTEMPTS):
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=[
            Part.from_uri(
                file_uri="https://storage.googleapis.com/cloud-samples-data/generative-ai/image/a-man-and-a-dog.png",
                mime_type="image/png",
            ),
            "Write a short and engaging blog post based on this image.",
        ],
    )

    cached_token_count = response.usage_metadata.cached_content_token_count or 0

    print(f"#{i + 1} Attempt")
    print(f"Input tokens: {response.usage_metadata.prompt_token_count}")
    print(f"Cached tokens: {cached_token_count}")
    print(f"Output tokens: {response.usage_metadata.candidates_token_count}")
    print(f"Total tokens: {response.usage_metadata.total_token_count}")
    print()

    if cached_token_count > 0:
        print(response.usage_metadata.cache_tokens_details)
        print("Cached content found, exiting loop.")
        break

#1 Attempt
Input tokens: 2334
Cached tokens: 0
Output tokens: 243
Total tokens: 3655

#2 Attempt
Input tokens: 2334
Cached tokens: 2004
Output tokens: 255
Total tokens: 4147

[ModalityTokenCount(
  modality=<MediaModality.IMAGE: 'IMAGE'>,
  token_count=1994
), ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=10
)]
Cached content found, exiting loop.


## Explicit caching

Using the explicit caching feature, you can pass some content to the model once, cache the input tokens, and then refer to the cached tokens for subsequent requests. The minimum input token count for context caching is 2,048 for all models.

### Create a context cache

Create a `CachedContent` object specifying the prompt you want to use, including the file and other fields you wish to cache.

**Notes**
- Caches are model specific. You cannot use a cache made with a different model as their tokenization might be slightly different.
- The default expiration time of a context cache is 60 minutes. You can specify a different expiration time using the `ttl` (time to live) or the `expire_time` property.

This example shows how to create a context cache using two large research papers stored in a Cloud Storage bucket, and set the `ttl` to 600s.

- Paper 1: [Gemini: A Family of Highly Capable Multimodal Models](https://arxiv.org/abs/2312.11805)
- Paper 2: [Gemini 1.5: Unlocking multimodal understanding across millions of tokens of context](https://arxiv.org/abs/2403.05530)

In [9]:
system_instruction = """
You are an expert researcher who has years of experience in conducting systematic literature surveys and meta-analyses of different topics.
You pride yourself on incredible accuracy and attention to detail. You always stick to the facts in the sources provided, and never make up new facts.
Now look at the research paper below, and answer the following questions in 1-2 sentences.
"""

cached_content = client.caches.create(
    model=MODEL_ID,
    config=CreateCachedContentConfig(
        contents=[
            Content(
                role="user",
                parts=[
                    Part.from_uri(
                        file_uri="gs://cloud-samples-data/generative-ai/pdf/2312.11805v3.pdf",
                        mime_type="application/pdf",
                    ),
                    Part.from_uri(
                        file_uri="gs://cloud-samples-data/generative-ai/pdf/2403.05530.pdf",
                        mime_type="application/pdf",
                    ),
                ],
            )
        ],
        system_instruction=system_instruction,
        ttl="600s",
    ),
)

You can access the properties of the cached content as example below. You can use its `name` or `resource_name` to reference the contents of the context cache.

**Note**: The `name` of the context cache is also referred to as cache ID.

In [10]:
print(cached_content.name)
print(cached_content.model)
print(cached_content.create_time)
print(cached_content.expire_time)
print(cached_content.usage_metadata)

projects/456822750436/locations/us-central1/cachedContents/4605639700907032576
projects/bdc-trainings/locations/us-central1/publishers/google/models/gemini-2.5-flash
2026-04-23 04:08:36.339946+00:00
2026-04-23 04:18:36.327198+00:00
audio_duration_seconds=None image_count=167 text_count=321 total_token_count=43163 video_duration_seconds=None


### Retrieve a context cache

You can use the property `name` to reference the contents of the context cache. For example:

In [11]:
new_cached_content = client.caches.get(name=cached_content.name)

### Use a context cache

To use the context cache, you provide the `cached_content` resource name in the `config` parameter of the `generate_content()` method.

Then you can query the model with a prompt, and the cached content will be used as a prefix to the prompt.

In [12]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents="What is the research goal shared by these research papers?",
    config=GenerateContentConfig(
        cached_content=cached_content.name,
    ),
)

display(Markdown(response.text))

The research goal is to introduce the Gemini family of multimodal models, aiming for strong generalist capabilities across image, audio, video, and text understanding. A key objective is to achieve cutting-edge understanding and reasoning performance in each domain, advancing the state of the art in various benchmarks.

You can check cached_content_token_count in the usage metadata which indicates how many tokens in the request were cached.

In [13]:
print(f"Input tokens: {response.usage_metadata.prompt_token_count}")
print(f"Cached tokens: {response.usage_metadata.cached_content_token_count or 0}")
print(f"Output tokens: {response.usage_metadata.candidates_token_count}")
print(f"Total tokens: {response.usage_metadata.total_token_count}")

Input tokens: 43173
Cached tokens: 43163
Output tokens: 57
Total tokens: 43762


### Use context caching in chat

You can use the context cache in a multi-turn chat session.


In [14]:
chat = client.chats.create(
    model=MODEL_ID,
    config=GenerateContentConfig(
        cached_content=cached_content.name,
    ),
)

In [15]:
prompt = """
How do the approaches to responsible AI development and mitigation strategies in Gemini 1.5 evolve from those in Gemini 1.0?
"""

response = chat.send_message(prompt)

display(Markdown(response.text))

The approaches to responsible AI deployment and mitigation in Gemini 1.5 Pro are mostly consistent with Gemini 1.0, but its impact assessment evolved to address the additional consequences of long-context understanding across modalities. A key mitigation update in Gemini 1.5 Pro is the incorporation of new image-to-text SFT data, as text-only SFT was found less effective for harm-inducing image-to-text queries.

In [16]:
prompt = """
Given the advancements presented in Gemini 1.5, what are the key future research directions identified in both papers
for further improving multimodal AI models?
"""

response = chat.send_message(prompt)

print(response.text)

Both papers emphasize the continued need for more robust and challenging evaluation methodologies to truly measure model understanding and complex reasoning, particularly for multimodal and long-context scenarios. Future research also focuses on reducing hallucinations, improving the reliability of model outputs, and further enhancing high-level reasoning abilities and multilingual capabilities, especially for low-resource languages.


### Update the expiration time of a context cache

The default expiration time of a context cache is 60 minutes. To update the expiration time, update one of the following properties:

`ttl` - The number of seconds that the cache lives after it's created or after the `ttl` is updated before it expires.

`expire_time` - A Timestamp that specifies the absolute date and time when the context cache expires.

In [17]:
cached_content = client.caches.update(
    name=cached_content.name,
    config=CreateCachedContentConfig(
        system_instruction=system_instruction,
        ttl="3600s",
    ),
)

print(cached_content.expire_time)

Type mismatch in _UpdateCachedContentParameters.config: expected UpdateCachedContentConfig, got CreateCachedContentConfig


2026-04-23 05:10:23.495275+00:00


### Delete a context cache

You can remove content from the cache using the delete operation.

In [18]:
client.caches.delete(name=cached_content.name)

DeleteCachedContentResponse(
  sdk_http_response=HttpResponse(
    headers=<dict len=9>
  )
)

## Next Steps

- Learn more about context caching on the [context caching overview](https://cloud.google.com/vertex-ai/generative-ai/docs/context-cache/context-cache-overview) page.